In [2]:
pip install pandas scikit-learn sentence-transformers joblib

In [55]:
from google.colab import files

uploaded = files.upload()

Saving intentions.csv to intentions.csv


In [29]:
import os

os.listdir("/content")

['.config',
 'intents.json',
 'intentions.csv',
 '.ipynb_checkpoints',
 'intents (1).json',
 'intent_classifier.pkl',
 'sample_data']

In [56]:
import pandas as pd

df = pd.read_csv("/content/intentions.csv")

print(df.head())
print()
print(df["intention"].value_counts())

                                        texte       intention
0  Pouvez-vous me présenter cette formation ?  FORMATION_INFO
1          En quoi consiste cette formation ?  FORMATION_INFO
2       Cette filière concerne quel domaine ?  FORMATION_INFO
3         À quoi correspond cette formation ?  FORMATION_INFO
4   Quel est le principe de cette formation ?  FORMATION_INFO

intention
RECOMMANDATION                194
FORMATION_COMPETENCES         123
CLARIFICATION                 117
COMPARAISON_FORMATIONS        113
FORMATION_PREREQUIS           113
ANALYSE_PROFIL                112
FORMATION_ADMISSION           109
EXPLICATION_RECOMMANDATION    106
FORMATION_DEBOUCHES           105
FORMATION_MATIERES            101
FORMATION_INFO                 99
intention                       2
Name: count, dtype: int64


Séparation du label et target

In [44]:
X = df["texte"]
y = df["intention"]

Séparation des données de test et entrainemnet

In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Model d'embedding + Vectorisation des données

In [58]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [59]:
X_train_embeddings = embedding_model.encode(
    X_train.tolist(),
    show_progress_bar=True
)

X_test_embeddings = embedding_model.encode(
    X_test.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Entrainement

In [60]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(
    max_iter=2000
)

classifier.fit(
    X_train_embeddings,
    y_train
)

LogisticRegression(max_iter=2000)

Tester le modèle

In [61]:
predictions = classifier.predict(
    X_test_embeddings
)

In [62]:
predictions = classifier.predict(
    X_test_embeddings
)

In [63]:
import joblib

joblib.dump(
    classifier,
    "intent_classifier.pkl"
)

['intent_classifier.pkl']

PREDICT.PY

In [64]:
import joblib
from sentence_transformers import SentenceTransformer


embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

classifier = joblib.load(
    "intent_classifier.pkl"
)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [67]:

question = input("Posez votre question : ")

embedding = embedding_model.encode(
    [question]
)

probabilities = classifier.predict_proba(embedding)[0]
classes = classifier.classes_

results = sorted(
    zip(classes, probabilities),
    key=lambda x: x[1],
    reverse=True
)

for intent, probability in results:
    print(f"{intent}: {probability:.4f}")

Posez votre question : Quelles compétences on acquiert en étudiant le droit
FORMATION_COMPETENCES: 0.7377
HORS_SCOPE: 0.1485
COMPARAISON_FORMATIONS: 0.0375
ANALYSE_PROFIL: 0.0191
FORMATION_DEBOUCHES: 0.0163
FORMATION_MATIERES: 0.0149
FORMATION_PREREQUIS: 0.0083
RECOMMANDATION: 0.0082
FORMATION_ADMISSION: 0.0053
intention: 0.0020
CLARIFICATION: 0.0014
EXPLICATION_RECOMMANDATION: 0.0007
FORMATION_INFO: 0.0002
